<a href="https://colab.research.google.com/github/Emily-als/worklens-llm/blob/feature%2Fdata-preprocessing/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets

In [2]:
import pandas as pd
import numpy as np

from datasets import load_dataset

SEED = 42

dataset = load_dataset(
    "lallantop/glassdoor",
    split="train"
)

df = dataset.to_pandas()

print("Original dataset shape:", df.shape)

archive (1).zip: reconstructing file:   0%|          |  0.00B / 88.0MB            

archive (1).zip: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/838566 [00:00<?, ? examples/s]

Original dataset shape: (838566, 18)


In [3]:
text_columns = ["headline", "pros", "cons"]

target_columns = [
    "work_life_balance",
    "comp_benefits",
    "senior_mgmt"
]

relevant_columns = ["firm"] + text_columns + target_columns

df_clean = df[relevant_columns].copy()

display(df_clean.head())

,firm,headline,pros,cons,work_life_balance,comp_benefits,senior_mgmt
0,AFH-Wealth-Management,"Young colleagues, poor micro management",Very friendly and welcoming to new staff. Easy...,"Poor salaries, poor training and communication.",4.0,3.0,3.0
1,AFH-Wealth-Management,"Excellent staff, poor salary","Friendly, helpful and hard-working colleagues",Poor salary which doesn't improve much with pr...,3.0,1.0,4.0
2,AFH-Wealth-Management,"Low salary, bad micromanagement",Easy to get the job even without experience in...,"Very low salary, poor working conditions, very...",1.0,1.0,1.0
3,AFH-Wealth-Management,Over promised under delivered,Nice staff to work with,No career progression and salary is poor,2.0,2.0,3.0
4,AFH-Wealth-Management,client reporting admin,"Easy to get the job, Nice colleagues.","Abysmal pay, around minimum wage. No actual tr...",2.0,1.0,1.0


In [4]:
valid_ratings = df_clean[target_columns].isin(
    [1, 2, 3, 4, 5]
).all(axis=1)

df_clean = df_clean.loc[valid_ratings].copy()

print("Remaining reviews:", len(df_clean))
print("Removed reviews:", len(df) - len(df_clean))

Remaining reviews: 679549
Removed reviews: 159017


In [5]:
## Duplicate and Conflicting Reviews Analysis

In [6]:
rating_mapping = {
    1: "Low",
    2: "Low",
    3: "Medium",
    4: "High",
    5: "High"
}

label_columns = []

for col in target_columns:
    label_col = col + "_label"
    df_clean[label_col] = df_clean[col].map(rating_mapping)
    label_columns.append(label_col)

In [7]:
duplicate_mask = df_clean.duplicated(
    subset=text_columns,
    keep=False
)

duplicate_reviews = df_clean.loc[duplicate_mask]

duplicate_groups = duplicate_reviews.groupby(text_columns)

conflicting_groups = (
    duplicate_groups[label_columns]
    .nunique()
    .gt(1)
    .any(axis=1)
)

print("Duplicate review groups:", len(conflicting_groups))

print("Groups with conflicting labels:",
      conflicting_groups.sum())

print("Groups with consistent labels:",
      (~conflicting_groups).sum())

Duplicate review groups: 28516
Groups with conflicting labels: 14
Groups with consistent labels: 28502


In [8]:
## Duplicate Reviews Removal

In [9]:
before_count = len(df_clean)

cross_company_groups = (
    duplicate_groups["firm"].nunique() > 1
)

print("Duplicate groups across companies:",
      cross_company_groups.sum())

Duplicate groups across companies: 9


In [10]:
excluded_groups = conflicting_groups | cross_company_groups

excluded_keys = excluded_groups[excluded_groups].index

duplicate_keys = pd.MultiIndex.from_frame(
    duplicate_reviews[text_columns]
)

excluded_indices = duplicate_reviews.index[
    duplicate_keys.isin(excluded_keys)
]

df_clean = df_clean.drop(index=excluded_indices)

print("Removed ambiguous reviews:",
      len(excluded_indices))

Removed ambiguous reviews: 39


In [11]:
df_clean = df_clean.drop_duplicates(
    subset=text_columns,
    keep="first"
).reset_index(drop=True)

print("Reviews before:", before_count)
print("Reviews after:", len(df_clean))
print("Total removed:", before_count - len(df_clean))

print("Remaining duplicates:",
      df_clean.duplicated(subset=text_columns).sum())

Reviews before: 679549
Reviews after: 650920
Total removed: 28629
Remaining duplicates: 0


In [12]:
## Model Input Preparation

In [13]:
## Construct Review Input Text
# Handle missing text fields before concatenation.

df_clean["input_text"] = (
    "Headline: " + df_clean["headline"].fillna("") + "\n"
    + "Pros: " + df_clean["pros"].fillna("") + "\n"
    + "Cons: " + df_clean["cons"].fillna("")
)

print(df_clean["input_text"].iloc[0])

Headline: Young colleagues, poor micro management
Pros: Very friendly and welcoming to new staff. Easy going ethic.
Cons: Poor salaries, poor training and communication.


In [14]:
## Validate Review Input Text

assert df_clean["input_text"].notna().all()
assert df_clean["input_text"].is_unique

print("Total reviews:", len(df_clean))
print("Missing texts:", df_clean["input_text"].isna().sum())
print("Duplicate texts:", df_clean["input_text"].duplicated().sum())

Total reviews: 650920
Missing texts: 0
Duplicate texts: 0


In [15]:
df_clean["is_mixed"] = (
    df_clean[label_columns].eq("High").any(axis=1) &
    df_clean[label_columns].eq("Low").any(axis=1)
)

print("Mixed reviews:", df_clean["is_mixed"].sum())

print("Mixed percentage:",
      round(df_clean["is_mixed"].mean() * 100, 2))

Mixed reviews: 146786
Mixed percentage: 22.55


In [16]:
df_clean["label_combination"] = (
    df_clean[label_columns[0]] + "_" +
    df_clean[label_columns[1]] + "_" +
    df_clean[label_columns[2]]
)

display(df_clean["label_combination"].value_counts())

,count
label_combination,
High_High_High,164337
Low_Low_Low,58558
High_Medium_High,35241
High_High_Medium,33363
Medium_High_High,33058
Medium_Medium_Medium,28132
Low_Medium_Low,24413
High_Medium_Medium,23485
Medium_High_Medium,20045


In [17]:
## Unseen Companies Holdout

In [18]:
HOLDOUT_FRACTION = 0.15

company_counts = df_clean["firm"].value_counts()
companies = sorted(company_counts.index)

n_holdout = round(len(companies) * HOLDOUT_FRACTION)

rng = np.random.default_rng(SEED)

best_error = float("inf")

for _ in range(100):
    candidate = rng.choice(
        companies,
        size=n_holdout,
        replace=False
    )

    fraction = (
        company_counts.loc[candidate].sum()
        / len(df_clean)
    )

    error = abs(fraction - HOLDOUT_FRACTION)

    if error < best_error:
        best_error = error
        unseen_companies = candidate

In [19]:
unseen_mask = df_clean["firm"].isin(unseen_companies)

df_unseen = df_clean.loc[unseen_mask].copy()
df_seen = df_clean.loc[~unseen_mask].copy()

print("Unseen companies:", df_unseen["firm"].nunique())
print("Unseen reviews:", len(df_unseen))
print("Unseen percentage:",
      round(len(df_unseen) / len(df_clean) * 100, 2))

print("Remaining reviews:", len(df_seen))

Unseen companies: 64
Unseen reviews: 97981
Unseen percentage: 15.05
Remaining reviews: 552939


In [20]:
overlap = set(df_seen["firm"]) & set(df_unseen["firm"])

assert len(overlap) == 0

print("Overlapping companies:", len(overlap))

Overlapping companies: 0


In [21]:
## Stratified Train-Validation-Test Split

In [22]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(
    df_seen,
    test_size=0.30,
    random_state=SEED,
    stratify=df_seen["label_combination"]
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=df_temp["label_combination"]
)

print("Train:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test))
print("Unseen companies:", len(df_unseen))

Train: 387057
Validation: 82941
Test: 82941
Unseen companies: 97981


In [23]:
all_indices = (
    df_train.index
    .append(df_val.index)
    .append(df_test.index)
    .append(df_unseen.index)
)

assert all_indices.is_unique
assert len(all_indices) == len(df_clean)

print("No overlapping reviews between splits.")
print("All reviews accounted for.")

No overlapping reviews between splits.
All reviews accounted for.


In [24]:
## Dataset Split Analysis

In [25]:
splits = {
    "Train": df_train,
    "Validation": df_val,
    "Test": df_test,
    "Unseen": df_unseen
}

summary = pd.DataFrame([
    {
        "Split": name,
        "Reviews": len(data),
        "Companies": data["firm"].nunique(),
        "Mixed Reviews": data["is_mixed"].sum(),
        "Mixed Percentage": round(
            data["is_mixed"].mean() * 100, 2
        )
    }
    for name, data in splits.items()
])

display(summary)

,Split,Reviews,Companies,Mixed Reviews,Mixed Percentage
0,Train,387057,364,86755,22.41
1,Validation,82941,356,18590,22.41
2,Test,82941,357,18590,22.41
3,Unseen,97981,64,22851,23.32


In [26]:
distribution = pd.DataFrame({
    name: data["label_combination"].value_counts(
        normalize=True
    ) * 100
    for name, data in splits.items()
}).fillna(0).round(2)

display(distribution)

,Train,Validation,Test,Unseen
label_combination,,,,
High_High_High,25.11,25.11,25.11,26.01
High_High_Low,2.44,2.44,2.44,3.17
High_High_Medium,4.93,4.93,4.93,6.26
High_Low_High,1.63,1.63,1.63,1.90
High_Low_Low,2.26,2.26,2.26,3.68
High_Low_Medium,1.72,1.72,1.72,2.58
High_Medium_High,5.29,5.29,5.29,6.12
High_Medium_Low,1.98,1.98,1.98,3.16
High_Medium_Medium,3.36,3.36,3.37,4.98


In [27]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [28]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/LLM Project")

assert PROJECT_DIR.is_dir(), "Project folder not found"

DATA_DIR = PROJECT_DIR / "DATASETS" / "REAL" / "Glassdoor" / "v1"

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [29]:
for name, data in splits.items():
    file_path = DATA_DIR / f"{name.lower()}.parquet"

    data.rename_axis("review_id").reset_index().to_parquet(
        file_path,
        index=False
    )

    print(f"Saved {name}: {len(data):,} reviews")

Saved Train: 387,057 reviews
Saved Validation: 82,941 reviews
Saved Test: 82,941 reviews
Saved Unseen: 97,981 reviews


In [30]:
summary.to_csv(
    DATA_DIR / "split_summary.csv",
    index=False
)

distribution.to_csv(
    DATA_DIR / "label_distribution.csv"
)

pd.Series(
    sorted(unseen_companies),
    name="firm"
).to_csv(
    DATA_DIR / "unseen_companies.csv",
    index=False
)

In [31]:
for name, data in splits.items():
    file_path = DATA_DIR / f"{name.lower()}.parquet"

    saved_data = pd.read_parquet(
        file_path,
        columns=["review_id"]
    )

    assert len(saved_data) == len(data)

    print(f"{name}: verified")

Train: verified
Validation: verified
Test: verified
Unseen: verified
